# AI by Intuition：三维教学 Transformer 的一次训练与 20 步轨迹

这个 Notebook 调用仓库中的**同一份三维参数表**；`pytorch_starter.py` 用 PyTorch 自动求导核对一步更新，`teaching_trace.py` 用 NumPy 参考实现观察 20 步的损失与贪心预测。

期望的一步结果：`1.798430 → 1.611377`，形状 `(3, 5)`，99 个可训练参数；第 20 步损失约 `0.483173`。

仅为同一条教学序列的训练记录，不代表真实语言泛化能力。

先填下面的 `BOOK_COMMIT`，再从上到下运行。Notebook 页面所在提交与实际运行代码提交分别核对；启动成功时会打印实际 SHA 和依赖版本。未填写或无法解析时立即停止，不回退到最新版本。

实验使用 CPU、float64；固定 NumPy/PyTorch 版本，数值断言容差为 `1e-8`，目标是复现到教材显示精度。


In [ ]:
# 填写本 PDF 版权页的提交号（7–40 位），不要填 main。
BOOK_COMMIT = ""

from pathlib import Path
import re
import subprocess
import sys


def prepare_checkout(commit, destination="ai-by-intuition", remote="https://github.com/CoCoThink/ai-by-intuition.git"):
    if not re.fullmatch(r"[0-9a-fA-F]{7,40}", commit):
        raise ValueError("请把 PDF 版权页的 Git 提交号填入 BOOK_COMMIT，然后重新运行。")
    destination = Path(destination).resolve()
    def git(*args):
        return subprocess.check_output(["git", "-C", str(destination), *args], text=True).strip()
    if not destination.exists():
        subprocess.run(["git", "clone", "--no-checkout", "--filter=blob:none", remote, str(destination)], check=True)
    else:
        if git("remote", "get-url", "origin") != remote:
            raise ValueError("已有目录的来源不同，请另选 destination。")
        if git("status", "--porcelain"):
            raise ValueError("已有目录含未提交修改，请先保留修改或另选 destination。")
        args = ["fetch", "origin"]
        if git("rev-parse", "--is-shallow-repository") == "true":
            args.append("--unshallow")
        subprocess.run(["git", "-C", str(destination), *args], check=True)
    try:
        resolved = git("rev-parse", "--verify", commit + "^{commit}")
    except subprocess.CalledProcessError as error:
        raise ValueError("无法解析该提交号；请核对版权页及仓库历史。") from error
    subprocess.run(["git", "-C", str(destination), "checkout", "--detach", resolved], check=True)
    actual = git("rev-parse", "HEAD")
    assert actual == resolved and actual.lower().startswith(commit.lower())
    print("实际运行代码 SHA:", actual)
    return destination


In [ ]:
checkout = prepare_checkout(BOOK_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(checkout / "requirements-teaching.txt")], check=True)
import platform
import numpy as np
import torch
print("Python:", platform.python_version(), "NumPy:", np.__version__, "PyTorch:", torch.__version__)
print("设备: CPU；精度: float64；比较容差: 1e-8")


In [ ]:
subprocess.run([sys.executable, str(checkout / "scripts/pytorch_starter.py")], check=True, cwd=checkout)
subprocess.run([sys.executable, str(checkout / "scripts/teaching_trace.py")], check=True, cwd=checkout)


## 可验证的结果

对照教材三位置的目标 `[4,5,3]`：初始贪心 `[2,4,3]`，一次更新后 `[4,4,3]`，第 9 步开始成为 `[4,5,3]`。

你可以修改 `scripts/teaching_reference.py` 中的 `LR` 做学习率实验，但改变参数后**不要**继续声称上述固定数字仍成立，也不要将新结果写回原书标准基线。
